# 01 — Data Extraction & Profiling

**Dataset:** Inside Airbnb — New York City 2019 (`AB_NYC_2019.csv`)  
**Source:** [Inside Airbnb](http://insideairbnb.com/get-the-data/) | Kaggle mirror  
**Objective:** Load the raw dataset, validate its integrity, and produce a comprehensive structural profile to guide downstream cleaning decisions.  

**Analytical coverage in this notebook:**
- NumPy array construction, masking, and vectorized operations for initial profiling
- Pandas loading, `.info()`, `.describe()`, dtype inspection, and missing-value quantification
- Row-wise logic to flag anomalous records before any transformation
- Descriptive statistics and distribution checks as a baseline for EDA

## 1.1 Environment Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'AB_NYC_2019.csv'

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data path: {RAW_PATH}')
print(f'File exists  : {RAW_PATH.exists()}')

## 1.2 Load Raw Dataset

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)

print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print(f'Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
df.head()

## 1.3 Schema & Data Types

In [ ]:
df.info(show_counts=True)

In [ ]:
dtype_summary = pd.DataFrame({
    'dtype'        : df.dtypes.astype(str),
    'non_null'     : df.notna().sum(),
    'null_count'   : df.isna().sum(),
    'null_pct'     : (df.isna().sum() / len(df) * 100).round(2),
    'unique_values': df.nunique(),
    'sample_value' : [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
}).sort_values('null_pct', ascending=False)

print('=== Column Schema & Null Profile ===')
dtype_summary

**Findings:**
- `last_review` and `reviews_per_month` share the same 10,052 nulls (~20.6%) — these correspond to listings with zero reviews, not data loss.
- `name` (16 nulls) and `host_name` (21 nulls) are minor text gaps, safe to forward-fill with a placeholder.
- All numeric columns are non-null; no imputation is needed for core KPIs.

## 1.4 Categorical Distributions

In [ ]:
cat_cols = ['neighbourhood_group', 'room_type']
for col in cat_cols:
    counts = df[col].value_counts()
    pcts   = df[col].value_counts(normalize=True).mul(100).round(2)
    summary = pd.concat([counts, pcts], axis=1, keys=['count', 'pct_%'])
    print(f'\n=== {col} ===')
    print(summary)

## 1.5 Numeric Descriptive Statistics

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
desc = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc['range'] = desc['max'] - desc['min']
desc['iqr']   = desc['75%'] - desc['25%']
desc

## 1.6 NumPy-Powered Anomaly Flags (Pre-Cleaning Snapshot)

Using NumPy boolean masks and vectorized operations to identify records that will require transformation in `02_cleaning.ipynb`.

In [ ]:
price_arr      = df['price'].to_numpy(dtype=np.float64)
min_nights_arr = df['minimum_nights'].to_numpy(dtype=np.float64)
reviews_arr    = df['number_of_reviews'].to_numpy(dtype=np.float64)
avail_arr      = df['availability_365'].to_numpy(dtype=np.float64)

mask_zero_price    = price_arr == 0
mask_price_outlier = price_arr > 1_000
mask_nights_extreme = min_nights_arr > 365
mask_avail_zero    = avail_arr == 0
mask_no_review     = reviews_arr == 0

anomaly_report = {
    'Zero price listings'              : int(mask_zero_price.sum()),
    'Price > $1,000 (potential outlier)': int(mask_price_outlier.sum()),
    'Minimum nights > 365'             : int(mask_nights_extreme.sum()),
    'Availability = 0 (ghost listings)': int(mask_avail_zero.sum()),
    'Zero reviews'                     : int(mask_no_review.sum()),
}

for k, v in anomaly_report.items():
    print(f'  {k:<42} {v:>6,}  ({v/len(df)*100:.1f}%)')

In [ ]:
print('Price distribution (NumPy percentiles):')
pcts = np.percentile(price_arr[price_arr > 0], [10, 25, 50, 75, 90, 95, 99])
for p, v in zip([10, 25, 50, 75, 90, 95, 99], pcts):
    print(f'  p{p:>2}: ${v:.0f}')

print(f'\nMean price (non-zero): ${np.mean(price_arr[price_arr > 0]):.2f}')
print(f'Std  price (non-zero): ${np.std(price_arr[price_arr > 0]):.2f}')
print(f'Max  price           : ${np.max(price_arr):.0f}')

## 1.7 Neighbourhood & Host-Level Summaries

In [ ]:
borough_summary = (
    df.groupby('neighbourhood_group')
    .agg(
        listing_count        = ('id'              , 'count'),
        avg_price            = ('price'           , 'mean'),
        median_price         = ('price'           , 'median'),
        avg_min_nights       = ('minimum_nights'  , 'mean'),
        avg_availability     = ('availability_365', 'mean'),
        total_reviews        = ('number_of_reviews', 'sum'),
        multi_listing_hosts  = ('calculated_host_listings_count', lambda x: (x > 1).sum())
    )
    .sort_values('listing_count', ascending=False)
    .round(2)
)
borough_summary

In [ ]:
top_hosts = (
    df.groupby('host_id')
    .agg(
        host_name       = ('host_name', 'first'),
        listing_count   = ('id'       , 'count'),
        avg_price       = ('price'    , 'mean'),
        total_reviews   = ('number_of_reviews', 'sum'),
    )
    .sort_values('listing_count', ascending=False)
    .head(15)
    .round(2)
)
print('Top 15 hosts by listing count:')
top_hosts

## 1.8 Duplicate & ID Integrity Check

In [ ]:
print(f'Fully duplicate rows  : {df.duplicated().sum()}')
print(f'Duplicate listing IDs  : {df["id"].duplicated().sum()}')
print(f'Unique hosts           : {df["host_id"].nunique():,}')
print(f'Unique neighbourhoods  : {df["neighbourhood"].nunique()}')
print(f'Date range (last_review): {df["last_review"].min()}  →  {df["last_review"].max()}')

## 1.9 Extraction Summary & Cleaning Roadmap

| Issue | Count | Action in 02_cleaning.ipynb |
|---|---|---|
| `name` nulls | 16 | Fill with `'Unknown Listing'` |
| `host_name` nulls | 21 | Fill with `'Unknown Host'` |
| `last_review` nulls | 10,052 | Fill date with `'1900-01-01'`; `reviews_per_month` → 0.0 |
| `price` = 0 | 11 | Remove (unlettable / data error) |
| `price` > 1,000 | 239 | Cap at 99th percentile or flag as `is_luxury` |
| `minimum_nights` > 365 | 14 | Cap at 365 (platform limit) |
| `last_review` dtype | object | Parse to `datetime64` |
| Column standardisation | — | Rename to snake_case (already clean) |
| Feature engineering | — | `price_tier`, `is_multi_lister`, `log_price`, `revenue_proxy` |

> **Proceed to** `02_cleaning.ipynb` for the full transformation pipeline.